# P29 — Kaggle One-Click Dual-Head Attribution Experiment

이 노트북은 **Kaggle에서 Run All 한 번으로 끝나도록** 만든 독립 실행본입니다.

실행 조건:
- Kaggle Internet: ON
- Accelerator: GPU
- 별도 업로드/패치 셀 없음
- `SEALED_HOLDOUT_V2` 미포함
- 모델/checkpoint/cache는 결과 ZIP에 저장하지 않음

실험 조건:
1. `SEMANTIC_ONLY_P20VIEW`
2. `DUAL_HEAD_MASKED_ABOUTNESS`
3. `DUAL_HEAD_RESIDUAL`

Checkpoint 선택은 기존 P20/P27과 동일하게:
`0.70 × NATURAL_DEV balanced accuracy + 0.30 × generator-dev Macro-F1`

최종 산출물:
`/kaggle/working/pii_p29_results.zip`


In [ ]:
# 1) Runtime / Kaggle preflight
import os,sys,subprocess,platform
print('python:',sys.version)
print('platform:',platform.platform())
try: print(subprocess.check_output(['nvidia-smi'],text=True)[:4000])
except Exception as e: print('GPU not visible:',e)
RUN_ROOT='/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
print('RUN_ROOT =',RUN_ROOT)


In [ ]:
# 2) Kaggle-safe pinned dependencies
# Pin the tokenizer stack explicitly. The notebook bypasses the model-specific tokenizer loader.
!python -m pip install -q "pandas==2.2.2" "scikit-learn==1.7.2"
!python -m pip install -q "transformers==4.57.1" "tokenizers==0.22.1" "huggingface_hub==0.36.0" sentencepiece "safetensors>=0.4,<1" requests urllib3

import pandas as _pd, sklearn as _sk, torch as _torch, transformers as _tf, tokenizers as _tks, huggingface_hub as _hf
print('pandas',_pd.__version__)
print('scikit-learn',_sk.__version__)
print('torch',_torch.__version__,'cuda',_torch.cuda.is_available())
print('transformers',_tf.__version__)
print('tokenizers',_tks.__version__)
print('huggingface_hub',_hf.__version__)
if not _torch.cuda.is_available():
    raise RuntimeError('GPU가 없습니다. Kaggle Settings > Accelerator에서 GPU를 활성화한 뒤 세션을 재시작하세요.')


In [ ]:
# 3) Robust model bootstrap — direct HTTPS download, then fully offline Transformers
# This deliberately bypasses huggingface_hub's shared HTTP client.
import os, time, pathlib, requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

MODEL_REPO = "team-lucid/deberta-v3-xsmall-korean"
MODEL_REVISION = "655178a346098a6eb6b7bf7f227056b70cc6d3c0"
MODEL_ROOT = pathlib.Path(RUN_ROOT) / "hf_models" / "deberta-v3-xsmall-korean-655178a"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_FILES = {
    "config.json": 100,
    "model.safetensors": 100_000_000,
    "tokenizer.json": 100_000,
    "tokenizer_config.json": 100,
    "special_tokens_map.json": 50,
}

def _good(path, min_bytes):
    return path.exists() and path.is_file() and path.stat().st_size >= min_bytes

def _download_direct(filename, min_bytes):
    dst = MODEL_ROOT / filename
    if _good(dst, min_bytes):
        print("cached:", filename, dst.stat().st_size)
        return dst

    url = f"https://huggingface.co/{MODEL_REPO}/resolve/{MODEL_REVISION}/{filename}?download=true"
    tmp = dst.with_suffix(dst.suffix + ".part")
    if tmp.exists():
        tmp.unlink()

    retry = Retry(
        total=8,
        connect=8,
        read=8,
        status=8,
        backoff_factor=1.5,
        status_forcelist=(408, 429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    session = requests.Session()
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.headers.update({"User-Agent":"p29-kaggle-direct-download/1.0"})

    last = None
    for attempt in range(1, 5):
        try:
            print(f"download {filename} attempt {attempt} ...")
            with session.get(url, stream=True, timeout=(30, 180), allow_redirects=True) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            f.write(chunk)
            if not _good(tmp, min_bytes):
                raise RuntimeError(f"downloaded file too small: {filename} size={tmp.stat().st_size if tmp.exists() else -1}")
            os.replace(tmp, dst)
            print("downloaded:", filename, dst.stat().st_size)
            return dst
        except Exception as e:
            last = e
            print("download failed:", repr(e))
            if tmp.exists():
                tmp.unlink()
            time.sleep(min(2 ** attempt, 20))
        finally:
            session.close()
    raise RuntimeError(f"Failed to download {filename} after retries: {last!r}")

for _name, _min_size in MODEL_FILES.items():
    _download_direct(_name, _min_size)

# Validate the exact local snapshot before allowing any training code to run.
missing = [name for name, min_size in MODEL_FILES.items() if not _good(MODEL_ROOT/name, min_size)]
if missing:
    raise RuntimeError(f"Local model snapshot incomplete: {missing}")

# From this point onward Transformers must never call the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
LOCAL_BASE = str(MODEL_ROOT)
TOKENIZER_JSON = str(MODEL_ROOT / "tokenizer.json")
print("LOCAL_BASE =", LOCAL_BASE)

# Load the serialized Rust tokenizer directly, then wrap it with the generic
# Transformers fast-tokenizer interface. This bypasses model-specific conversion.
from tokenizers import Tokenizer as RustTokenizer
from tokenizers.processors import TemplateProcessing
from transformers import AutoConfig, AutoModel, PreTrainedTokenizerFast

def load_local_tokenizer():
    backend = RustTokenizer.from_file(TOKENIZER_JSON)
    cls_id = backend.token_to_id("[CLS]")
    sep_id = backend.token_to_id("[SEP]")
    if cls_id is None or sep_id is None:
        raise RuntimeError(f"tokenizer.json is missing required DeBERTa specials: CLS={cls_id}, SEP={sep_id}")
    # Force the canonical DeBERTa sequence format instead of trusting legacy metadata.
    backend.post_processor = TemplateProcessing(
        single="[CLS] $A [SEP]",
        pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[("[CLS]", cls_id), ("[SEP]", sep_id)],
    )
    tok = PreTrainedTokenizerFast(
        tokenizer_object=backend,
        bos_token="[CLS]",
        eos_token="[SEP]",
        unk_token="[UNK]",
        sep_token="[SEP]",
        pad_token="[PAD]",
        cls_token="[CLS]",
        mask_token="[MASK]",
        model_max_length=512,
        clean_up_tokenization_spaces=False,
    )
    tok.model_input_names = ["input_ids", "token_type_ids", "attention_mask"]
    return tok

# ---- tokenizer + model preflight: must pass BEFORE any training ----
_cfg = AutoConfig.from_pretrained(LOCAL_BASE, local_files_only=True)
_tok = load_local_tokenizer()
for attr in ["pad_token_id","cls_token_id","sep_token_id","unk_token_id","mask_token_id"]:
    if getattr(_tok, attr) is None:
        raise RuntimeError(f"Tokenizer special token missing: {attr}")
_single = _tok("홍길동은 서울에서 진료를 받았다.", add_special_tokens=True)
_pair = _tok("홍길동은 병원에 방문했다.", "이 기록은 다른 사람의 것이다.", add_special_tokens=True)
if not _single.get("input_ids") or not _pair.get("input_ids"):
    raise RuntimeError("Direct tokenizer produced empty input_ids")
if _single["input_ids"][0] != _tok.cls_token_id or _single["input_ids"][-1] != _tok.sep_token_id:
    raise RuntimeError(f"Single-sequence special-token pattern invalid: {_single['input_ids'][:4]} ... {_single['input_ids'][-4:]}")
if _pair["input_ids"][0] != _tok.cls_token_id or _pair["input_ids"].count(_tok.sep_token_id) != 2:
    raise RuntimeError("Pair-sequence DeBERTa special-token pattern invalid")
if "token_type_ids" in _pair and 1 not in _pair["token_type_ids"]:
    raise RuntimeError("Pair token_type_ids did not mark the second sequence")
_max_id = max(_single["input_ids"] + _pair["input_ids"])
if _max_id >= int(_cfg.vocab_size):
    raise RuntimeError(f"Tokenizer/model vocab mismatch: max_id={_max_id}, config_vocab={_cfg.vocab_size}")

# Full forward smoke test catches model/tokenizer incompatibility immediately.
_model = AutoModel.from_pretrained(LOCAL_BASE, local_files_only=True)
_enc = _tok("홍길동의 검사 결과는 정상이다.", return_tensors="pt", truncation=True, max_length=64)
_allowed = {k:v for k,v in _enc.items() if k in {"input_ids","attention_mask","token_type_ids"}}
with _torch.inference_mode():
    _out = _model(**_allowed)
if _out.last_hidden_state.shape[0] != 1 or _out.last_hidden_state.shape[1] != _allowed["input_ids"].shape[1]:
    raise RuntimeError(f"Unexpected model forward shape: {_out.last_hidden_state.shape}")
print("DIRECT TOKENIZER PREFLIGHT: PASS", type(_tok).__name__, "vocab", len(_tok))
print("LOCAL MODEL FORWARD PREFLIGHT: PASS", tuple(_out.last_hidden_state.shape))
del _cfg, _tok, _single, _pair, _max_id, _model, _enc, _allowed, _out
import gc as _gc
_gc.collect()


In [ ]:
# 3) Exact target corpus + Frozen Gold v1 (embedded, unchanged)
import base64,gzip,json
TARGET_B64='...'
GOLD_B64='...'
def dec(s):
 raw=gzip.decompress(base64.b64decode(s)).decode("utf-8")
 return [json.loads(x) for x in raw.splitlines() if x.strip()]
TARGET=dec(TARGET_B64);GOLD=dec(GOLD_B64)
print("target",len(TARGET),"gold",len(GOLD))


In [ ]:
# 4) Common setup
import re,random,gc,time,copy,math,os
import numpy as np,pandas as pd,torch
import torch.nn as nn
from collections import Counter,defaultdict,OrderedDict
from torch.utils.data import Dataset,DataLoader
from transformers import AutoModel,get_linear_schedule_with_warmup
from sklearn.metrics import precision_recall_fscore_support,accuracy_score,confusion_matrix

BASE=LOCAL_BASE
assert os.path.isdir(BASE) and os.path.isfile(os.path.join(BASE,"config.json")), BASE
SEED=26026
MAX_LEN=256
EPOCHS=5
LR=2e-5
WD=.01
BATCH=16

LABELS=["CLEAN","DETECT","REVIEW"]
L2I={x:i for i,x in enumerate(LABELS)}

TRAIN=[r for r in TARGET if r["split"]=="train"]
GEN_DEV=[r for r in TARGET if r["split"]=="dev"]

def seed_all(s=SEED):
 random.seed(s);np.random.seed(s);torch.manual_seed(s)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(s)

def met(g,p):
 pr,rc,f,_=precision_recall_fscore_support(g,p,labels=LABELS,zero_division=0)
 return {
  "n":len(g),
  "accuracy":float(accuracy_score(g,p)),
  "macro_f1":float(np.mean(f)),
  "by_class":{l:{"precision":float(pr[i]),"recall":float(rc[i]),"f1":float(f[i])}
              for i,l in enumerate(LABELS)},
  "confusion":confusion_matrix(g,p,labels=LABELS).tolist(),
 }


In [ ]:
# Model/tokenizer-loading invariant: everything is local from this point onward.
assert os.environ.get("HF_HUB_OFFLINE") == "1"
assert os.environ.get("TRANSFORMERS_OFFLINE") == "1"
assert os.path.isdir(BASE), BASE
assert os.path.exists(os.path.join(BASE,"model.safetensors"))
assert os.path.exists(TOKENIZER_JSON)
_probe=load_local_tokenizer()
assert _probe.convert_tokens_to_ids("[CLS]") == _probe.cls_token_id
assert _probe.convert_tokens_to_ids("[SEP]") == _probe.sep_token_id
print("OFFLINE MODEL INVARIANT: PASS")
print("DIRECT TOKENIZER INVARIANT: PASS",type(_probe).__name__)
del _probe


In [ ]:
# 9) Frozen Gold marker-complete slices
EVAL=[]
for i,r in enumerate(GOLD):
 text=r.get("normalized_text","")
 if "<TARGET>" not in text or "<SENS>" not in text:continue
 EVAL.append({**r,"_row_index":i})
print("marker-complete",len(EVAL),Counter(r.get("tier") for r in EVAL))

SLICES={
 "core":[r for r in EVAL if r.get("tier")=="core_behavioral"],
 "main":[r for r in EVAL if r.get("tier") in ("core_behavioral","stress_composite","legal_boundary")],
 "stress":[r for r in EVAL if r.get("tier")=="stress_composite"],
 "legal_boundary":[r for r in EVAL if r.get("tier")=="legal_boundary"],
 "natural_dev":[r for r in EVAL if r.get("tier")=="external_independent"],
 "natural_sensitive":[r for r in EVAL if r.get("tier")=="external_independent" and r.get("score_scope")=="external_sensitive_mini"],
 "natural_surface":[r for r in EVAL if r.get("tier")=="external_independent" and r.get("score_scope")=="external_surface"],
 "pair_swap":[r for r in EVAL if r.get("tier")=="pair_swap"],
}

def finish_result(name,hist,bestep,eval_fn):
 res={"experiment":name,"best_epoch":bestep,"history":hist,"slices":{},"predictions":{}}
 for sn,rr in SLICES.items():
  m,p,pr=eval_fn(rr)
  res["slices"][sn]=m
  res["predictions"][sn]=[
   {"id":r.get("id",str(i)),"gold":r["pair_label"],"pred":pp,
    "probs":dict(zip(LABELS,map(float,q))),"family":r.get("family"),"category":r.get("category")}
   for i,(r,pp,q) in enumerate(zip(rr,p,pr))
  ]
 core={x["id"]:x["pred"] for x in res["predictions"]["core"]}
 swap={x["id"]:x["pred"] for x in res["predictions"]["pair_swap"]}
 a=[];b=[];both=[]
 for r in SLICES["pair_swap"]:
  src=(r.get("source") or {}).get("from");rid=r.get("id")
  if src in core and rid in swap:
   aa=core[src]=="CLEAN";bb=swap[rid]=="DETECT"
   a.append(aa);b.append(bb);both.append(aa and bb)
 res["pair_completeness"]={"n":len(both),"original_clean":float(np.mean(a)),
                           "swapped_detect":float(np.mean(b)),"both":float(np.mean(both))}
 return res


In [ ]:
# 8) Build NLI -> balanced Binary-RE intermediate encoder once
import requests
from transformers import AutoModelForSequenceClassification

KLUE_ROOT="https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark"
def getj(url):
 r=requests.get(url,timeout=180);r.raise_for_status();return r.json()

NLI_T=getj(KLUE_ROOT+"/klue-nli-v1.1/klue-nli-v1.1_train.json")
RE_T=getj(KLUE_ROOT+"/klue-re-v1.1/klue-re-v1.1_train.json")
RE_D=getj(KLUE_ROOT+"/klue-re-v1.1/klue-re-v1.1_dev.json")

base_tok=load_local_tokenizer()
ORIG_BASE_VOCAB=len(base_tok)
NLI2I={"entailment":0,"neutral":1,"contradiction":2}

class NLIAuxDS(Dataset):
 def __init__(self,rows):self.rows=rows
 def __len__(self):return len(self.rows)
 def __getitem__(self,i):
  r=self.rows[i]
  e=base_tok(r["premise"],r["hypothesis"],truncation=True,max_length=MAX_LEN,padding=False)
  e["labels"]=NLI2I[r["gold_label"]]
  return e

def insert_re_markers(x):
 s=x["sentence"]
 ins=[(int(x["subject_entity"]["start_idx"]),"<SUBJ>"),
      (int(x["object_entity"]["start_idx"]),"<OBJ>")]
 for pos,m in sorted(ins,reverse=True):s=s[:pos]+m+s[pos:]
 return s

aux_tok=load_local_tokenizer()
aux_tok.add_special_tokens({"additional_special_tokens":["<SUBJ>","<OBJ>"]})

rng=random.Random(SEED)
neg=[x for x in RE_T if x["label"]=="no_relation"]
pos=[x for x in RE_T if x["label"]!="no_relation"]
rng.shuffle(neg);rng.shuffle(pos);n=min(len(neg),len(pos))
BIN_T=[(x,0) for x in neg[:n]]+[(x,1) for x in pos[:n]]
rng.shuffle(BIN_T)
BIN_D=[(x,0 if x["label"]=="no_relation" else 1) for x in RE_D]

class BinAuxDS(Dataset):
 def __init__(self,rows):self.rows=rows
 def __len__(self):return len(self.rows)
 def __getitem__(self,i):
  r,y=self.rows[i]
  e=aux_tok(insert_re_markers(r),truncation=True,max_length=MAX_LEN,padding=False)
  e["labels"]=y
  return e

def aux_collate(tok_):
 def fn(batch):
  y=torch.tensor([x.pop("labels") for x in batch],dtype=torch.long)
  z=tok_.pad(batch,padding=True,return_tensors="pt");z["labels"]=y
  return z
 return fn

def train_aux_epoch(model,ds,tok_,batch_size=24,lr=LR):
 dl=DataLoader(ds,batch_size=batch_size,shuffle=True,collate_fn=aux_collate(tok_))
 opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=WD)
 sch=get_linear_schedule_with_warmup(opt,max(1,int(.1*len(dl))),len(dl))
 scaler=torch.amp.GradScaler("cuda");model.cuda().train();ls=[]
 for b in dl:
  b={k:v.cuda() for k,v in b.items()};opt.zero_grad(set_to_none=True)
  with torch.amp.autocast("cuda"):loss=model(**b).loss
  scaler.scale(loss).backward();scaler.unscale_(opt)
  torch.nn.utils.clip_grad_norm_(model.parameters(),1)
  scaler.step(opt);scaler.update();sch.step();ls.append(float(loss.detach().cpu()))
 return float(np.mean(ls))

@torch.inference_mode()
def eval_bin_aux(model,rows):
 dl=DataLoader(BinAuxDS(rows),batch_size=48,shuffle=False,collate_fn=aux_collate(aux_tok))
 g=[];p=[];model.eval()
 for b in dl:
  y=b.pop("labels");b={k:v.cuda() for k,v in b.items()}
  z=model(**b).logits.argmax(-1).cpu()
  g+=y.tolist();p+=z.tolist()
 cr=sum(a==0 and q==0 for a,q in zip(g,p))/max(1,sum(a==0 for a in g))
 dr=sum(a==1 and q==1 for a,q in zip(g,p))/max(1,sum(a==1 for a in g))
 return (cr+dr)/2

seed_all()
nli_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=3,ignore_mismatched_sizes=True,local_files_only=True)
nli_loss=train_aux_epoch(nli_model,NLIAuxDS(NLI_T),base_tok,24)
NLI_STATE={k:v.detach().cpu().clone() for k,v in nli_model.base_model.state_dict().items()}
print("NLI aux loss",nli_loss)
del nli_model;torch.cuda.empty_cache();gc.collect()

seed_all()
bin_model=AutoModelForSequenceClassification.from_pretrained(BASE,num_labels=2,ignore_mismatched_sizes=True,local_files_only=True)
bin_model.resize_token_embeddings(len(aux_tok))
# Transfer NLI state. Original-vocab weights match; new marker embeddings stay initialized.
current=bin_model.base_model.state_dict()
with torch.no_grad():
 for k,v in NLI_STATE.items():
  if k not in current:continue
  if current[k].shape==v.shape:
   current[k].copy_(v)
  elif current[k].ndim==2 and v.ndim==2 and current[k].shape[1]==v.shape[1] and current[k].shape[0]>=v.shape[0]:
   current[k][:min(ORIG_BASE_VOCAB,v.shape[0])].copy_(v[:min(ORIG_BASE_VOCAB,v.shape[0])])
bin_model.base_model.load_state_dict(current)

best_state=None;best_bal=-1;BIN_META=[]
for ep in range(2):
 loss=train_aux_epoch(bin_model,BinAuxDS(BIN_T),aux_tok,24)
 bal=eval_bin_aux(bin_model,BIN_D)
 BIN_META.append({"epoch":ep+1,"loss":loss,"balanced_accuracy":bal})
 print("BINRE aux",BIN_META[-1])
 if bal>best_bal:
  best_bal=bal
  # Store only the original base-vocabulary-compatible encoder state.
  st=bin_model.base_model.state_dict()
  best_state={k:v.detach().cpu().clone() for k,v in st.items()}

NLI_BIN_STATE=best_state
del bin_model;torch.cuda.empty_cache();gc.collect()
print("NLI->BinaryRE intermediate ready",best_bal)


In [ ]:
# 11) P29 shared input view + masked aboutness supervision
import torch.nn.functional as F

P29_SEED=29029
ABOUT_WEIGHT=0.35
RESIDUAL_SCALE=0.35

P29_SPECIAL=["<TARGET>","<OTHER>","<SENS>","<SUBJ>","<OBJ>","[GAP]"]
p29_tok=load_local_tokenizer()
p29_tok.add_special_tokens({"additional_special_tokens":P29_SPECIAL})

SUBJ_ID=p29_tok.convert_tokens_to_ids("<SUBJ>")
OBJ_ID=p29_tok.convert_tokens_to_ids("<OBJ>")
if SUBJ_ID==p29_tok.unk_token_id or OBJ_ID==p29_tok.unk_token_id:
    raise RuntimeError("P29 pair markers were not added as tokens")

def p29_pair_mark(text):
    # P20-compatible view: preserve original semantic markers and prepend pair anchors.
    return text.replace("<TARGET>","<SUBJ><TARGET>").replace("<SENS>","<OBJ><SENS>")

def p29_split_units(text):
    out=[]
    for line in text.splitlines():
        line=line.strip()
        if line:
            out += [p.strip() for p in re.split(r'(?<=[.!?。])\s+',line) if p.strip()]
    return out

def p29_bundle(text,max_len=MAX_LEN):
    ids=p29_tok(text,add_special_tokens=True,truncation=False)["input_ids"]
    if len(ids)<=max_len:
        return text
    units=p29_split_units(text)
    scores=[]
    for u in units:
        sc=0
        if "<OBJ>" in u or "<SENS>" in u: sc+=100
        if "<SUBJ>" in u or "<TARGET>" in u: sc+=80
        if "<OTHER>" in u: sc+=35
        if any(x in u for x in ["[MAP]","[IDMAP]","[DATA]","[RESULT]","[Sheet:","ID=","id=",
                                 "subject=","대상자","당사자","명의자","[ROW]","[HEADER]"]): sc+=35
        if any(x in u for x in ["최종","정정","오류","누락","미확인","unresolved","실제"]): sc+=12
        scores.append(sc)
    mandatory={i for i,s in enumerate(scores) if s>=35}
    chosen=set(mandatory)
    for i in list(mandatory):
        if i>0: chosen.add(i-1)
        if i+1<len(units): chosen.add(i+1)
    for i in sorted(range(len(units)),key=lambda j:(scores[j],-j),reverse=True):
        if i in chosen: continue
        trial="\n[GAP]\n".join(units[j] for j in sorted(chosen|{i}))
        if len(p29_tok(trial,add_special_tokens=True,truncation=False)["input_ids"])<=max_len:
            chosen.add(i)
    out="\n[GAP]\n".join(units[i] for i in sorted(chosen))
    if "<SUBJ>" not in out or "<OBJ>" not in out:
        raise RuntimeError("P29 bundle lost the primary pair markers")
    return out

def p29_row_text_label(r):
    if "text" in r and "label" in r:
        return r["text"],r["label"]
    if "normalized_text" in r and "pair_label" in r:
        return r["normalized_text"],r["pair_label"]
    raise KeyError(f"Unsupported row schema: {sorted(r.keys())}")

ABOUT_FAMILIES={
    "mixed_language","multi_entity","other_person","possessive_agent",
    "quotation","reversal","table_serialized",
}

def swap_target_other(text):
    return text.replace("<TARGET>","<TMP_TARGET>").replace("<OTHER>","<TARGET>").replace("<TMP_TARGET>","<OTHER>")

BASE_TRAIN=[]
AUX_ROWS=[]
_seen_aux=set()

for r in TRAIN:
    rr=dict(r)
    rr["_action_mask"]=1
    rr["_about_mask"]=0
    rr["_about_label"]=0

    eligible=(
        r.get("family") in ABOUT_FAMILIES
        and r.get("label") in ("CLEAN","DETECT")
        and "<OTHER>" in r.get("text","")
    )
    if eligible:
        rr["_about_mask"]=1
        rr["_about_label"]=1 if r["label"]=="DETECT" else 0

        new_text=swap_target_other(r["text"])
        new_about=1-rr["_about_label"]
        key=(new_text,new_about)
        if key not in _seen_aux:
            _seen_aux.add(key)
            ar=dict(r)
            ar["id"]=str(r.get("id",""))+"-ABOUTSWAP"
            ar["text"]=new_text
            ar["label"]="CLEAN"  # dummy only; action loss is masked
            ar["_action_mask"]=0
            ar["_about_mask"]=1
            ar["_about_label"]=new_about
            AUX_ROWS.append(ar)

    BASE_TRAIN.append(rr)

if len(BASE_TRAIN)!=len(TRAIN):
    raise RuntimeError("BASE_TRAIN size mismatch")
if len(AUX_ROWS)!=324:
    raise RuntimeError(f"Expected 324 aboutness-only swap rows, got {len(AUX_ROWS)}")
if Counter(r["_about_label"] for r in AUX_ROWS)!={0:162,1:162}:
    raise RuntimeError(f"Unexpected aux balance: {Counter(r['_about_label'] for r in AUX_ROWS)}")

DUAL_TRAIN=BASE_TRAIN+AUX_ROWS
print("P29 supervision:",{
    "action_rows":sum(r["_action_mask"] for r in BASE_TRAIN),
    "original_about_rows":sum(r["_about_mask"] for r in BASE_TRAIN),
    "aux_about_rows":len(AUX_ROWS),
    "dual_train_rows":len(DUAL_TRAIN),
})
print("P29 AUXILIARY SUPERVISION INTEGRITY: PASS")

class P29Dataset(Dataset):
    def __init__(self,rows):
        self.rows=rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        text,label=p29_row_text_label(r)
        text=p29_bundle(p29_pair_mark(text))
        e=p29_tok(text,truncation=True,max_length=MAX_LEN,padding=False)
        subj=[j for j,x in enumerate(e["input_ids"]) if x==SUBJ_ID]
        obj=[j for j,x in enumerate(e["input_ids"]) if x==OBJ_ID]
        if not subj or not obj:
            raise RuntimeError(f"Pair marker missing at row {r.get('id',i)}")
        e["labels"]=L2I[label]
        e["ss"]=subj[0]
        e["os"]=obj[0]
        e["action_mask"]=int(r.get("_action_mask",1))
        e["about_mask"]=int(r.get("_about_mask",0))
        e["about_labels"]=int(r.get("_about_label",0))
        return e

def p29_collate(batch):
    labels=torch.tensor([x.pop("labels") for x in batch],dtype=torch.long)
    ss=torch.tensor([x.pop("ss") for x in batch],dtype=torch.long)
    oo=torch.tensor([x.pop("os") for x in batch],dtype=torch.long)
    action_mask=torch.tensor([x.pop("action_mask") for x in batch],dtype=torch.bool)
    about_mask=torch.tensor([x.pop("about_mask") for x in batch],dtype=torch.bool)
    about_labels=torch.tensor([x.pop("about_labels") for x in batch],dtype=torch.long)
    z=p29_tok.pad(batch,padding=True,return_tensors="pt")
    z["labels"]=labels
    z["ss"]=ss
    z["os"]=oo
    z["action_mask"]=action_mask
    z["about_mask"]=about_mask
    z["about_labels"]=about_labels
    return z


In [ ]:
# 12) Single-encoder semantic + aboutness heads
def transfer_nli_bin_state(encoder):
    current=encoder.state_dict()
    copied=0
    with torch.no_grad():
        for k,v in NLI_BIN_STATE.items():
            if k not in current:
                continue
            dst=current[k]
            if dst.shape==v.shape:
                dst.copy_(v)
                copied+=v.numel()
            elif dst.ndim==2 and v.ndim==2 and dst.shape[1]==v.shape[1]:
                n=min(ORIG_BASE_VOCAB,dst.shape[0],v.shape[0])
                dst[:n].copy_(v[:n])
                copied+=n*dst.shape[1]
    encoder.load_state_dict(current)
    return copied

class P29Model(nn.Module):
    def __init__(self,mode):
        super().__init__()
        if mode not in ("semantic","dual","residual"):
            raise ValueError(mode)
        self.mode=mode
        self.encoder=AutoModel.from_pretrained(BASE,local_files_only=True)
        copied=transfer_nli_bin_state(self.encoder)
        self.encoder.resize_token_embeddings(len(p29_tok))
        h=self.encoder.config.hidden_size
        self.action_head=nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(h,h),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(h,3),
        )
        self.about_head=nn.Sequential(
            nn.Linear(2*h,h),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(h,2),
        )
        self._copied=copied

    def forward(self,input_ids,attention_mask,ss,os,labels=None,
                action_mask=None,about_mask=None,about_labels=None,
                token_type_ids=None,**kwargs):
        enc={"input_ids":input_ids,"attention_mask":attention_mask}
        if token_type_ids is not None:
            enc["token_type_ids"]=token_type_ids
        hidden=self.encoder(**enc).last_hidden_state
        idx=torch.arange(input_ids.size(0),device=input_ids.device)

        semantic_logits=self.action_head(hidden[:,0])
        pair_feat=torch.cat([hidden[idx,ss],hidden[idx,os]],dim=-1)
        about_logits=self.about_head(pair_feat)

        final_logits=semantic_logits
        if self.mode=="residual":
            margin=about_logits[:,1]-about_logits[:,0]
            residual=torch.stack([-margin,margin,torch.zeros_like(margin)],dim=-1)
            final_logits=semantic_logits + RESIDUAL_SCALE*residual

        loss=None
        if labels is not None:
            loss=final_logits.sum()*0.0
            if action_mask is None:
                action_mask=torch.ones_like(labels,dtype=torch.bool)
            if action_mask.any():
                loss=loss+F.cross_entropy(final_logits[action_mask],labels[action_mask])
            if self.mode!="semantic":
                if about_mask is not None and about_mask.any():
                    loss=loss+ABOUT_WEIGHT*F.cross_entropy(about_logits[about_mask],about_labels[about_mask])
        return loss,final_logits,about_logits

@torch.inference_mode()
def eval_p29(model,rows,bs=32):
    dl=DataLoader(P29Dataset(rows),batch_size=bs,shuffle=False,collate_fn=p29_collate)
    gold=[];pred=[];probs=[]
    model.eval()
    for b in dl:
        y=b.pop("labels")
        b={k:v.cuda() for k,v in b.items()}
        _,logits,_=model(**b)
        q=torch.softmax(logits.float().cpu(),dim=-1)
        gold += [LABELS[i] for i in y.tolist()]
        pred += [LABELS[i] for i in q.argmax(-1).tolist()]
        probs += q.tolist()
    return met(gold,pred),pred,probs

def natural_stats(m):
    cr=m["by_class"]["CLEAN"]["recall"]
    dr=m["by_class"]["DETECT"]["recall"]
    return {
        "clean_recall":float(cr),
        "detect_recall":float(dr),
        "balanced_accuracy":float((cr+dr)/2),
        "accuracy":float(m["accuracy"]),
    }

def train_p29(name,mode,train_rows):
    seed_all(P29_SEED)
    model=P29Model(mode).cuda()
    print(name,"transferred elements",model._copied,"train_rows",len(train_rows))

    dl=DataLoader(P29Dataset(train_rows),batch_size=BATCH,shuffle=True,collate_fn=p29_collate)
    enc_params=list(model.encoder.parameters())
    head_params=list(model.action_head.parameters())+list(model.about_head.parameters())
    opt=torch.optim.AdamW([
        {"params":enc_params,"lr":LR},
        {"params":head_params,"lr":1e-4},
    ],weight_decay=WD)

    total=len(dl)*EPOCHS
    sch=get_linear_schedule_with_warmup(opt,max(1,int(.1*total)),total)
    scaler=torch.amp.GradScaler("cuda")

    hist=[];best_score=-1;best_state=None;best_epoch=None
    for ep in range(EPOCHS):
        model.train();losses=[]
        for b in dl:
            b={k:v.cuda() for k,v in b.items()}
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss,_,_=model(**b)
            if not torch.isfinite(loss):
                raise RuntimeError(f"non-finite loss: {name} epoch={ep+1}")
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(opt);scaler.update();sch.step()
            losses.append(float(loss.detach().cpu()))

        gd,_,_=eval_p29(model,GEN_DEV)
        ndm,_,_=eval_p29(model,SLICES["natural_dev"])
        nd=natural_stats(ndm)
        score=.70*nd["balanced_accuracy"]+.30*gd["macro_f1"]
        row={
            "epoch":ep+1,
            "loss":float(np.mean(losses)),
            "generator_dev_macro_f1":float(gd["macro_f1"]),
            "natural_dev":nd,
            "selection_score":float(score),
        }
        hist.append(row)
        print(name,row)
        if score>best_score:
            best_score=score
            best_epoch=ep+1
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}

    if best_state is None:
        raise RuntimeError(f"No checkpoint selected for {name}")
    model.load_state_dict(best_state)
    model.cuda()
    return model,hist,best_epoch,best_score


In [ ]:
# 13) Full preflight before any expensive target training
probe_rows=BASE_TRAIN[:2]+AUX_ROWS[:2]
probe_batch=next(iter(DataLoader(P29Dataset(probe_rows),batch_size=4,shuffle=False,collate_fn=p29_collate)))
for mode in ("semantic","dual","residual"):
    seed_all(P29_SEED)
    m=P29Model(mode).cuda().eval()
    b={k:v.cuda() for k,v in probe_batch.items()}
    with torch.inference_mode():
        loss,logits,about=m(**b)
    if logits.shape!=(4,3) or about.shape!=(4,2):
        raise RuntimeError(f"P29 preflight shape failure {mode}: {logits.shape} {about.shape}")
    if not torch.isfinite(logits).all() or not torch.isfinite(about).all():
        raise RuntimeError(f"P29 preflight non-finite logits: {mode}")
    print("P29 PREFLIGHT",mode,"PASS","loss",float(loss.detach().cpu()))
    del m
    torch.cuda.empty_cache();gc.collect()
print("P29 FULL PREFLIGHT: PASS")


In [ ]:
# 14) Run exactly three P29 conditions
P29_RESULTS=[]
EXPERIMENTS=[
    ("SEMANTIC_ONLY_P20VIEW","semantic",BASE_TRAIN),
    ("DUAL_HEAD_MASKED_ABOUTNESS","dual",DUAL_TRAIN),
    ("DUAL_HEAD_RESIDUAL","residual",DUAL_TRAIN),
]

for name,mode,train_rows in EXPERIMENTS:
    t0=time.time()
    model,hist,bestep,bestscore=train_p29(name,mode,train_rows)
    res=finish_result(name,hist,bestep,lambda rr,m=model:eval_p29(m,rr))
    res["mode"]=mode
    res["best_selection_score"]=float(bestscore)
    res["train_n"]=len(train_rows)
    res["train_wall_seconds"]=float(time.time()-t0)
    P29_RESULTS.append(res)
    print("DONE",name,
          "core",res["slices"]["core"]["macro_f1"],
          "main",res["slices"]["main"]["macro_f1"],
          "natural",natural_stats(res["slices"]["natural_dev"]),
          "pair",res["pair_completeness"])
    del model
    torch.cuda.empty_cache();gc.collect()

actual=[r["experiment"] for r in P29_RESULTS]
expected=[x[0] for x in EXPERIMENTS]
assert len(actual)==3 and len(set(actual))==3 and actual==expected,(actual,expected)
print("P29 EXECUTION INTEGRITY: PASS",actual)


In [ ]:
# 15) Compare, diagnose, and save compact Kaggle result ZIP
import zipfile

P20_REF={
    "experiment":"P20_NLI_THEN_BINARY_RE",
    "core_f1":0.8157987734591344,
    "main_f1":0.7735082476066676,
    "natural_bal_acc":0.660633484162896,
    "natural_clean_recall":0.6153846153846154,
    "natural_detect_recall":0.7058823529411765,
    "pair_both":0.5726495726495726,
    "stress_f1":0.04597701149425287,
    "legal_f1":0.5384615384615384,
}

P28_TEACHER_REF={
    "experiment":"P28_P20_PLUS_PAIR_BASELINE_W045",
    "core_f1":0.8169239631509554,
    "main_f1":0.7760113449768622,
    "natural_bal_acc":0.6900452488687783,
    "natural_clean_recall":0.6153846153846154,
    "natural_detect_recall":0.7647058823529411,
    "pair_both":0.6410256410256411,
    "stress_f1":0.06666666666666667,
    "legal_f1":0.5384615384615384,
}

def summary_row(r):
    nat=r["slices"]["natural_dev"]
    cr=nat["by_class"]["CLEAN"]["recall"]
    dr=nat["by_class"]["DETECT"]["recall"]
    return {
        "experiment":r["experiment"],
        "core_f1":r["slices"]["core"]["macro_f1"],
        "main_f1":r["slices"]["main"]["macro_f1"],
        "natural_bal_acc":(cr+dr)/2,
        "natural_clean_recall":cr,
        "natural_detect_recall":dr,
        "pair_both":r["pair_completeness"]["both"],
        "pair_original_clean":r["pair_completeness"]["original_clean"],
        "pair_swapped_detect":r["pair_completeness"]["swapped_detect"],
        "stress_f1":r["slices"]["stress"]["macro_f1"],
        "legal_f1":r["slices"]["legal_boundary"]["macro_f1"],
        "best_epoch":r["best_epoch"],
        "selection_score":r["best_selection_score"],
        "train_n":r["train_n"],
        "wall_seconds":r["train_wall_seconds"],
    }

rows=[P20_REF,P28_TEACHER_REF]+[summary_row(r) for r in P29_RESULTS]
df=pd.DataFrame(rows)
display(df)

HARD_IDS=["EXTS-N1","EXTS-N2","EXTS-N3","EXTS-N4","EXTS-N5","EXTS-N6",
          "EXTS-N7","EXTS-N8","EXTS-N9","EXTS-N10","EXTR-B2","EXTR-P4"]
DIAG={}
for r in P29_RESULTS:
    fam=defaultdict(list)
    for x in r["predictions"]["core"]:
        fam[x.get("family")].append(x["gold"]==x["pred"])
    pm={x["id"]:x for x in r["predictions"]["natural_dev"]}
    DIAG[r["experiment"]]={
        "selection_history":r["history"],
        "worst_core_families":sorted(
            [{"family":k,"accuracy":float(np.mean(v))} for k,v in fam.items()],
            key=lambda x:x["accuracy"]
        )[:10],
        "hard_natural":[pm[i] for i in HARD_IDS if i in pm],
        "pair_completeness":r["pair_completeness"],
    }

OUT=os.path.join(RUN_ROOT,"p29_results")
os.makedirs(OUT,exist_ok=True)
comparison=os.path.join(OUT,"p29_comparison.csv")
results=os.path.join(OUT,"p29_results.json")
diagnostics=os.path.join(OUT,"p29_diagnostics.json")
meta=os.path.join(OUT,"p29_meta.json")

df.to_csv(comparison,index=False)
with open(results,"w",encoding="utf-8") as f:
    json.dump({
        "p20_reference":P20_REF,
        "p28_teacher_reference":P28_TEACHER_REF,
        "results":P29_RESULTS,
    },f,ensure_ascii=False,indent=2)
with open(diagnostics,"w",encoding="utf-8") as f:
    json.dump(DIAG,f,ensure_ascii=False,indent=2)
with open(meta,"w",encoding="utf-8") as f:
    json.dump({
        "seed":P29_SEED,
        "model_repo":MODEL_REPO,
        "model_revision":MODEL_REVISION,
        "model_load_mode":"direct-download + direct-tokenizer-json + offline-local-model",
        "selection_formula":"0.70*NATURAL_DEV_balanced_accuracy + 0.30*generator_dev_macro_f1",
        "about_weight":ABOUT_WEIGHT,
        "residual_scale":RESIDUAL_SCALE,
        "about_families":sorted(ABOUT_FAMILIES),
        "about_aux_rows":len(AUX_ROWS),
        "augmentation_uses_frozen":False,
        "augmentation_uses_natural_dev":False,
        "sealed_holdout_v2_opened":False,
        "note":"No model/checkpoint/cache files included.",
    },f,ensure_ascii=False,indent=2)

ALLOW=[comparison,results,diagnostics,meta]
zip_path=os.path.join(RUN_ROOT,"pii_p29_results.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED,compresslevel=9) as z:
    for path in ALLOW:
        if not os.path.isfile(path):
            raise RuntimeError(f"Missing output file: {path}")
        z.write(path,arcname=os.path.basename(path))

size=os.path.getsize(zip_path)
print("P29 RESULT ZIP:",zip_path)
print("size MB:",round(size/1024/1024,3))
if size>=20*1024*1024:
    raise RuntimeError("Result ZIP unexpectedly large; model/checkpoint may have leaked into output")
print("P29 RESULT ZIP INTEGRITY: PASS")


## 실행 완료 기준

마지막 셀에서 아래 문구가 나오면 완료입니다.

- `P29 FULL PREFLIGHT: PASS`
- `P29 EXECUTION INTEGRITY: PASS`
- `P29 RESULT ZIP INTEGRITY: PASS`

Kaggle 우측 **Files**에서 `/kaggle/working/pii_p29_results.zip`만 다운로드하면 됩니다.
